# Question B1 (15 marks)

Real world datasets often have a mix of numeric and categorical features – this dataset is one example. To build models on such data, categorical features have to be encoded or embedded.

PyTorch Tabular is a library that makes it very convenient to build neural networks for tabular data. It is built on top of PyTorch Lightning, which abstracts away boilerplate model training code and makes it easy to integrate other tools, e.g. TensorBoard for experiment tracking.

For questions B1 and B2, the following features should be used:   
- **Numeric / Continuous** features: dist_to_nearest_stn, dist_to_dhoby, degree_centrality, eigenvector_centrality, remaining_lease_years, floor_area_sqm
- **Categorical** features: month, town, flat_model_type, storey_range



---



In [1]:
!pip install pytorch_tabular[extra]

In [3]:
!pip show pytorch_tabular

Name: pytorch_tabular
Version: 1.2.0
Summary: A standard framework for using Deep Learning for tabular data
Home-page: 
Author: 
Author-email: Manu Joseph <manujosephv@gmail.com>
License: MIT
Location: C:\Users\Owner\anaconda3\Lib\site-packages
Requires: einops, numpy, omegaconf, pandas, pytorch-lightning, rich, scikit-base, scikit-learn, scipy, torch, torchmetrics
Required-by: 


In [5]:
SEED = 42

import os

import random
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd

from pytorch_tabular import TabularModel
from pytorch_tabular.models import CategoryEmbeddingModelConfig
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)

#### 1. Divide the dataset (‘hdb_price_prediction.csv’) into train, validation and test sets by using entries from year 2019 and before as training data, year 2020 as validation data and year 2021 as test data.
**Do not** use data from year 2022 and year 2023.



In [35]:
def process_data(df):
    # split by year
    train_df = df[df['year'] <= 2019]   # 2019 and before
    val_df   = df[df['year'] == 2020]   # 2020 only
    test_df  = df[df['year'] == 2021]   # 2021 only
    
    print(f"Train size : {len(train_df)}")
    print(f"Val size   : {len(val_df)}")
    print(f"Test size  : {len(test_df)}")
    
    # verify no 2022/2023 data
    print(f"\nYears in train : {sorted(train_df['year'].unique())}")
    print(f"Years in val   : {sorted(val_df['year'].unique())}")
    print(f"Years in test  : {sorted(test_df['year'].unique())}")
    
    return train_df, val_df, test_df

In [37]:
df = pd.read_csv('hdb_price_prediction.csv')

# check the date column
print(df.columns)
print(df.head())

Index(['month', 'year', 'town', 'full_address', 'nearest_stn',
       'dist_to_nearest_stn', 'dist_to_dhoby', 'degree_centrality',
       'eigenvector_centrality', 'flat_model_type', 'remaining_lease_years',
       'floor_area_sqm', 'storey_range', 'resale_price'],
      dtype='object')
   month  year        town              full_address   nearest_stn  \
0      1  2017  ANG MO KIO  406 ANG MO KIO AVENUE 10    Ang Mo Kio   
1      1  2017  ANG MO KIO   108 ANG MO KIO AVENUE 4    Ang Mo Kio   
2      1  2017  ANG MO KIO   602 ANG MO KIO AVENUE 5  Yio Chu Kang   
3      1  2017  ANG MO KIO  465 ANG MO KIO AVENUE 10    Ang Mo Kio   
4      1  2017  ANG MO KIO   601 ANG MO KIO AVENUE 5  Yio Chu Kang   

   dist_to_nearest_stn  dist_to_dhoby  degree_centrality  \
0             1.007264       7.006044           0.016807   
1             1.271389       7.983837           0.016807   
2             1.069743       9.090700           0.016807   
3             0.946890       7.519889           0.0

In [39]:
train_df, val_df, test_df = process_data(df)

Train size : 64057
Val size   : 23313
Test size  : 29057

Years in train : [2017, 2018, 2019]
Years in val   : [2020]
Years in test  : [2021]


#### 2. Refer to the documentation of **PyTorch Tabular** and perform the following tasks: https://pytorch-tabular.readthedocs.io/en/latest/#usage
- Use **[DataConfig](https://pytorch-tabular.readthedocs.io/en/latest/data/)** to define the target variable, as well as the names of the continuous and categorical variables.
- Use **[TrainerConfig](https://pytorch-tabular.readthedocs.io/en/latest/training/)** to automatically tune the learning rate. Set batch_size to be 1024 and set max_epoch as 50.
- Use **[CategoryEmbeddingModelConfig](https://pytorch-tabular.readthedocs.io/en/latest/models/#category-embedding-model)** to create a feedforward neural network with 1 hidden layer containing 50 neurons.
- Use **[OptimizerConfig](https://pytorch-tabular.readthedocs.io/en/latest/optimizer/)** to choose Adam optimiser. There is no need to set the learning rate (since it will be tuned automatically) nor scheduler.
- Use **[TabularModel](https://pytorch-tabular.readthedocs.io/en/latest/tabular_model/)** to initialise the model and put all the configs together.

In [43]:
# ── Check data types ───────────────────────────────────────────────
print(df.dtypes)
print("\n")

month                       int64
year                        int64
town                       object
full_address               object
nearest_stn                object
dist_to_nearest_stn       float64
dist_to_dhoby             float64
degree_centrality         float64
eigenvector_centrality    float64
flat_model_type            object
remaining_lease_years     float64
floor_area_sqm            float64
storey_range               object
resale_price              float64
dtype: object




In [45]:
# ── Categorical columns (object dtype) ────────────────────────────
categorical_cols = [
    'town',
    'full_address',
    'nearest_stn',
    'flat_model_type',
    'storey_range'
]

# ── Continuous columns (numeric dtype, excluding target/date) ──────
continuous_cols = [
    'dist_to_nearest_stn',
    'dist_to_dhoby',
    'degree_centrality',
    'eigenvector_centrality',
    'remaining_lease_years',
    'floor_area_sqm'
]

# ── Target ─────────────────────────────────────────────────────────
target_col = ['resale_price']

# ── Drop these (not features) ──────────────────────────────────────
# month, year → date columns, not features

print("Categorical :", categorical_cols)
print("Continuous  :", continuous_cols)
print("Target      :", target_col)

Categorical : ['town', 'full_address', 'nearest_stn', 'flat_model_type', 'storey_range']
Continuous  : ['dist_to_nearest_stn', 'dist_to_dhoby', 'degree_centrality', 'eigenvector_centrality', 'remaining_lease_years', 'floor_area_sqm']
Target      : ['resale_price']


In [47]:
# ── Step 2: DataConfig ─────────────────────────────────────────────
data_config = DataConfig(
    target        = target_col,
    continuous_cols  = continuous_cols,
    categorical_cols = categorical_cols,
)

In [49]:
# ── Step 3: TrainerConfig ──────────────────────────────────────────
trainer_config = TrainerConfig(
    batch_size        = 1024,
    max_epochs        = 50,
    auto_lr_find      = True,    # automatically tune learning rate
)

In [51]:
# ── Step 4: CategoryEmbeddingModelConfig ───────────────────────────
model_config = CategoryEmbeddingModelConfig(
    task         = 'regression',   # predicting price
    layers       = '50',           # 1 hidden layer with 50 neurons
    activation   = 'ReLU',
)

In [53]:
# ── Step 5: OptimizerConfig ────────────────────────────────────────
optimizer_config = OptimizerConfig(
    optimizer = 'Adam',            # no lr needed since auto_lr_find=True
)

In [64]:
# ── Step 6: TabularModel ───────────────────────────────────────────
model = TabularModel(
    data_config      = data_config,
    model_config     = model_config,
    optimizer_config = optimizer_config,
    trainer_config   = trainer_config,
)

2026-03-09 21:34:24,860 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off


In [66]:
model.fit(
    train      = train_df,
    validation = val_df
)

Seed set to 42
2026-03-09 21:34:27,541 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-03-09 21:34:27,596 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-03-09 21:34:27,750 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: CategoryEmbeddingModel
2026-03-09 21:34:27,807 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-03-09 21:34:27,893 - {pytorch_tabular.tabular_model:655} - INFO - Auto LR Find Started
C:\Users\Owner\anaconda3\Lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:881: Checkpoint directory C:\Users\Owner\Desktop\Assignment\saved

Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=100` reached.
Restoring states from the checkpoint path at C:\Users\Owner\Desktop\Assignment\.lr_find_68d07884-bb5d-4bb4-9537-28925a3cdc02.ckpt
C:\Users\Owner\anaconda3\Lib\site-packages\lightning_fabric\utilities\cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't ha

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ CategoryEmbeddingBackbone │  7.2 K │ train │     0 │
│ 1 │ _embedding_layer │ Embedding1dLayer          │  437 K │ train │     0 │
│ 2 │ head             │ LinearHead                │     51 │ train │     0 │
│ 3 │ loss             │ MSELoss                   │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 444 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 444 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 17                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

2026-03-09 21:35:20,389 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-03-09 21:35:20,389 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model
C:\Users\Owner\anaconda3\Lib\site-packages\pytorch_tabular\utils\python_utils.py:92: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for 

#### 3. Report the test MSE error and the test R2 value that you obtained.



In [69]:
# ── Evaluate on test set ───────────────────────────────────────────
result = model.evaluate(test_df)
print(result)

# ── Predict on test set ────────────────────────────────────────────
pred_df = model.predict(test_df)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

C:\Users\Owner\anaconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │      287999557632.0       │
│  test_mean_squared_error  │      287999557632.0       │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 287999557632.0, 'test_mean_squared_error': 287999557632.0}]


In [71]:
# ── Calculate MSE and R2 ───────────────────────────────────────────
from sklearn.metrics import mean_squared_error, r2_score

y_true = test_df['resale_price'].values
y_pred = pred_df['resale_price_prediction'].values

mse = mean_squared_error(y_true, y_pred)
r2  = r2_score(y_true, y_pred)

print(f"Test MSE : {mse:.4f}")
print(f"Test R2  : {r2:.4f}")

Test MSE : 287999588801.6956
Test R2  : -9.8876


4.Print out the corresponding rows in the dataframe for the top 20 test samples with the largest errors. Are there any patterns or common characteristics among these data points? Based on your observations, suggest and briefly explain a potential strategy to improve the model on these cases.



In [ ]:
# YOUR CODE & RESULT HERE